# 📊 Multi-City Analysis

Compare trip patterns across Oslo, Bergen, and Trondheim.
Topics: city comparison · top routes · weekend vs weekday · seasonal trends.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")
from utils import (
    load, load_app_ready, describe, feature_matrix,
    plot_hourly, plot_monthly, plot_top_stations, plot_duration_dist, plot_city_comparison,
    export_df, export_figure,
    CITIES, CITY_DISPLAY_MAP,
)
print("Workspace utils loaded ✓")

In [ ]:
# ── Load all cities, all years (silver layer) ─────────────────────────────────
YEAR = None   # int or None for all years

df = load(year=YEAR)
print(f"Total trips loaded: {len(df):,}")
print(df['city'].value_counts())

In [ ]:
# ── City comparison ───────────────────────────────────────────────────────────
fig_cities = plot_city_comparison(df, title=f"Total Trips per City ({YEAR or 'all years'})")

In [ ]:
# ── Top routes (start → end) ──────────────────────────────────────────────────
CITY_FILTER = "oslo"  # Set to a city name or None for all

route_df = df[df['city'] == CITY_FILTER] if CITY_FILTER else df

routes = (
    route_df
    .assign(route=route_df['start_station_name'].str.cat(route_df['end_station_name'], sep=' → '))
    ['route'].value_counts().head(15)
)

fig_routes, ax = plt.subplots(figsize=(10, 6))
ax.barh(routes.index[::-1], routes.values[::-1], color='#1f77b4', alpha=0.85)
ax.set(xlabel='Trips', title=f'Top 15 Routes — {CITY_FILTER or "All Cities"}')
plt.tight_layout()

In [ ]:
# ── Weekend vs Weekday ────────────────────────────────────────────────────────
import pandas as pd

df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
df['day_of_week'] = df['started_at'].dt.dayofweek
df['is_weekend'] = df['day_of_week'] >= 5

wk_counts = df.groupby(['city', 'is_weekend']).size().unstack(fill_value=0)
wk_counts.columns = ['Weekday', 'Weekend']
wk_counts['Weekend %'] = (wk_counts['Weekend'] / wk_counts.sum(axis=1) * 100).round(1)

print(wk_counts)

fig_weekend, ax = plt.subplots(figsize=(8, 4))
wk_counts[['Weekday', 'Weekend']].plot(kind='bar', ax=ax, colormap='tab10', alpha=0.85)
ax.set(xlabel='City', ylabel='Trips', title='Weekday vs Weekend Demand')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()

In [ ]:
# ── Seasonal trend per city ───────────────────────────────────────────────────
df['month'] = df['started_at'].dt.month
seasonal = df.groupby(['city', 'month']).size().unstack(level=0, fill_value=0)

fig_seasonal, ax = plt.subplots(figsize=(12, 4))
for city in seasonal.columns:
    ax.plot(seasonal.index, seasonal[city], marker='o', linewidth=2, label=city.title())
ax.set(xlabel='Month', ylabel='Trips', title='Monthly Seasonality by City',
       xticks=range(1, 13),
       xticklabels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.legend()
plt.tight_layout()

In [ ]:
# ── Duration statistics per city ──────────────────────────────────────────────
if 'duration_seconds' in df.columns:
    dur = df.copy()
    dur['duration_min'] = dur['duration_seconds'] / 60
    dur = dur[(dur['duration_min'] > 0) & (dur['duration_min'] <= 60)]
    stats = dur.groupby('city')['duration_min'].describe()[['mean','50%','std']]
    stats.columns = ['Mean (min)', 'Median (min)', 'Std (min)']
    print(stats.round(2))
else:
    print("No duration_seconds column — skipping duration stats")

In [ ]:
# ── Export key visuals to Streamlit app ───────────────────────────────────────
export_figure("analysis_city_comparison", fig_cities)
export_figure("analysis_top_routes",      fig_routes)
export_figure("analysis_weekend",         fig_weekend)
export_figure("analysis_seasonal",        fig_seasonal)
print("Exported ✓ — refresh Insights page in the app")